# Extracting, Transforming and Loading Data (ETL)

In [13]:
# Import libraries
import pandas as pd
from pathlib import Path

# List of years for which we want to load survey data.
YEARS = [2022, 2023, 2024]

# Resolve project root so this notebook works whether CWD is repo root or notebooks/.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data" / "raw").exists()), Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
print(f"Working directory: {Path.cwd()}")
print(f"Using RAW_DIR: {RAW_DIR}")

# Initialize an empty list to hold DataFrames for each year.
frames = []

Working directory: c:\Users\aliff\Desktop\Local Repo\salary-prediction\notebooks
Using RAW_DIR: c:\Users\aliff\Desktop\Local Repo\salary-prediction\data\raw


In [22]:
# Cell 0 — notebook settings
# Load the autoreload extension to automatically reload modules before executing code
%load_ext autoreload 
# Set autoreload mode to 2, which means all modules will be reloaded before executing code
%autoreload 2 

# Suppress warnings to keep the notebook output clean, especially during development when there may be deprecation warnings or other non-critical warnings that can be safely ignored.
import warnings
warnings.filterwarnings("ignore")
# Set the maximum number of columns to display when printing DataFrames to 50
pd.set_option("display.max_columns", 50) 
# Set the float format for displaying DataFrames to two decimal places
pd.set_option("display.float_format", "{:,.f}".format) 


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load and combine all years' data into a single DataFrame with a new 'survey_year' column to track the year of each row.

In [15]:
# Iterate through each year, read the corresponding CSV file, and append to the list.
for year in YEARS:
    path = RAW_DIR / str(year) / "survey_results_public.csv"
    
    # Check if file path exists. If not, raise FileNotFoundError
    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}. Run scripts/acquire.sh first to download/extract data."
        )

    # Read the CSV file into a DataFrame
    df = pd.read_csv(path, low_memory=False)
    df["survey_year"] = year # Add new column 'year' set to its year
    frames.append(df)        # Append the DataFrame to the list of frames for later concatenation.
    
    # Print the number of rows and columns for each year's DataFrame 
    print(f"{year}: {len(df):,} rows, {df.shape[1]} cols") 

2022: 73,268 rows, 80 cols
2023: 89,184 rows, 85 cols
2024: 65,437 rows, 115 cols


In [16]:
# After processing all years, concatenate the list of DataFrames into a single DataFrame. 
# The ignore_index=True option is used to reset the index in the combined DataFrame, 
# ensuring that it runs sequentially from 0 to n-1, 
# which is important for data integrity.
df_raw = pd.concat(frames, ignore_index=True)

## Inspect Extracted Data

In [ ]:
# Check the total number of rows and columns in the combined DataFrame
print(f"\nCombined: {len(df_raw):,} rows, {df_raw.shape[1]} cols")

In [ ]:
# Check the column names of the combined DataFrame
print("\nColumn names")
print(df_raw.columns.tolist())

In [ ]:
# Check the value counts of data types in the combined DataFrame to get an overview of the distribution of different data types, 
# which can help identify if there are any unexpected data types that may require attention during data cleaning or preprocessing.
print("\ndf_raw.dtypes value counts")
print(df_raw.dtypes.value_counts())

# Check the data types of each column in the combined DataFrame to understand the structure of the dataset and 
# identify any potential issues with data types that may need to be addressed during data cleaning or preprocessing.
print("\ndf_raw.dtypes")
print(df_raw.dtypes)


In [ ]:
# Check the first few rows of the combined DataFrame to verify 
# that the data has been loaded and combined correctly, 
print(df_raw.head()) 

In [ ]:
# Check if the 'ConvertedCompYearly' column exists in the combined DataFrame. 
# If it does, print summary statistics and counts of non-null and null values to understand the distribution of salary data in the dataset.
print(df_raw["ConvertedCompYearly"].describe())
print(f"Non-null salary rows: {df_raw['ConvertedCompYearly'].notna().sum():,}")
print(f"Null salary rows:     {df_raw['ConvertedCompYearly'].isna().sum():,}")


In [ ]:
# Check the distribution of non-null salary values by printing summary statistics including specific percentiles.
salary = df_raw["ConvertedCompYearly"].dropna()
print(salary.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

In [ ]:
# Inspect nulls by calculating the percentage of null values in each column, sorting them in descending order, and displaying the top 30 columns with null values. 
# This helps identify which features have the most missing data and may require attention during data cleaning or feature engineering.
null_pct = df_raw.isnull().mean().sort_values(ascending=False)
null_pct[null_pct > 0.0].head(30)